# 01 Data Collection - Literature Review and Source Validation

The 15-minute city is a compact way to ask a much older planning question: can everyday life be organized so that essential needs are reachable without depending on long motorized trips? Moreno and colleagues popularized the contemporary formulation around proximity, diversity, density, and digitalization, but the idea also connects to neighborhood-unit planning, mixed-use urbanism, transit-oriented development, and public-health research on walkability. For a Shanghai project, the concept is useful because it translates a broad quality-of-life claim into a measurable accessibility problem: from a residential location or grid cell, which daily services can be reached within a plausible 15-minute travel budget, and how evenly is that accessibility distributed across the metropolis?

The most important lesson from recent work is that a 15-minute city should not be treated as a simple count of amenities inside a circle. Accessibility depends on the transport mode, the street network, the quality of the destination, and the social meaning of the trip. A nearby park with a locked gate, a sports venue that charges high fees, or a supermarket across a hostile arterial road should not be interpreted in the same way as a welcoming public facility on a safe walking route. Studies of walkability and active travel therefore emphasize network distance, street connectivity, perceived safety, shade, cycling infrastructure, and the mix of land uses. Public-health research adds another layer: routine physical activity is more likely when sport and recreation opportunities are close, pleasant, affordable, and embedded in daily routes rather than isolated as special-purpose destinations.

The Paris heterogeneity paper supplied for this project is especially relevant because it warns against a one-size-fits-all 15-minute metric. It shows that urban areas with similar average accessibility may still differ strongly by neighborhood structure and local distribution of services. In other words, the citywide mean can hide the lived experience of peripheral towns, industrial edges, new towns, and dense central districts. That argument matters for Shanghai because the municipality contains very different urban forms: the historic core, Pudong's development corridors, suburban districts such as Qingpu and Fengxian, island and waterfront landscapes, and large patches of logistics or manufacturing land. A credible analysis should therefore keep the spatial unit small enough to reveal heterogeneity and should report scores by local cells rather than only by district averages.

For this Track A project, the baseline 15-minute city is defined as access to daily retail and food, primary healthcare, education or childcare, open space, public transport, and civic/community services. The health and sport track then asks whether a place supports a healthy lifestyle beyond minimum daily convenience. I operationalize that track with five components: sport facilities, green/outdoor access, cycling environment, healthy food access, and environmental quality. This follows the public-health logic that exercise is shaped by both formal facilities and informal environments. Gyms, courts, swimming pools, parks, green corridors, safe bikeable roads, and fresh-food retailers all contribute to the opportunity structure for healthy routines.

The data strategy reflects these concepts. Gaode POI shapefiles provide rich point-level amenities for sport, food, healthcare, education, civic services, and transport-related facilities. The Baidu AOI layer adds polygonal places, including parks, residential compounds, and a housing-price proxy used only in the web detail panel. The 2024 administrative boundary layer defines the citywide analysis mask and district labels so coastal districts, Changxing Island, and Chongming Island remain visible. The 2020 built-up area layer is retained as a source reference rather than as the final web-map mask. Traffic shapefiles supply bus stops, metro stations, metro exits, and related transit features. AI-interpreted green-space shapefiles and land-use polygons strengthen the open-space and environmental components. Finally, the simplified Shanghai road parquet derived from OSM-style network extraction contributes cycling and major-road indicators.

The project uses a 500 m grid as the working analysis surface and H3 resolution 8 as the web display layer. The 500 m grid is intuitive for urban planning because it approximates a fine neighborhood block scale while remaining computationally manageable. H3 hexagons are useful for the final application because they create equal-index spatial units that aggregate well, render efficiently, and avoid some visual bias of square grids. The notebooks model 15-minute access with mode-specific catchments: walking at about 1.2 km, cycling at about 3.5 km, transit as a local public-transport catchment, and car at 5 km. This is an accessibility proxy rather than a full routing model. The road network is used for cycling and environmental context, and the supplied network-extract notebook documents how a future extension could replace straight-line catchments with Dijkstra network isochrones.

The final score is deliberately transparent. For each grid cell, distance and nearby counts are combined into category scores from 0 to 100. The baseline score is a weighted average of six everyday-service categories. The Track A score is a weighted average of sport, green/outdoor, cycling, healthy food, and environmental quality. The composite score is 60% baseline and 40% Track A, reflecting that healthy lifestyle opportunities should build on, rather than replace, everyday urban accessibility. The resulting H3 GeoJSON is not a definitive judgment of neighborhoods; it is a reproducible analytical layer designed for comparison, critique, and interactive exploration.


---

# Setup steps

This notebook follows the style of the supplied urban-analytics examples: small cells, explicit setup, then repeated validation tables before any scoring. The raw data are intentionally not hidden behind one black-box function. Later notebooks use helper functions for reproducibility, but this first notebook opens the data catalogue and shows what each source contributes.


In [ ]:
from pathlib import Path
import os
import sys
import json
import textwrap
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / ".deps"))
sys.path.insert(0, str(ROOT))

ROOT


In [ ]:
import geopandas as gpd
import pyogrio
import pyarrow.parquet as pq

from scripts.build_outputs import (
    RAW,
    PROJECT_CRS,
    WGS84,
    BASELINE_WEIGHTS,
    TRACK_WEIGHTS,
    MODE_RADII_M,
    load_env,
    locate_paths,
)
from scripts.poi_2024 import load_poi_2024_points

load_env()
paths = locate_paths()


## Data sources used in this project

The table below is the conceptual source log. The following cells then validate the files on disk, including feature counts, geometry type, CRS, and selected fields.


In [ ]:
source_log = pd.DataFrame([
    {"family": "Administrative", "dataset": "2024 Shanghai city/district boundaries", "use": "citywide 500 m grid mask, district labels"},
    {"family": "Built area", "dataset": "2020 built-up area", "use": "source reference for urbanized land coverage"},
    {"family": "POI", "dataset": "Gaode WGS84 POI shapefiles", "use": "food, health, education, sport, civic, transit proxy"},
    {"family": "POI", "dataset": "POI 2024 classified CSVs", "use": "preferred Shanghai POI source for Track A sport, food, scenic, transit, and baseline education/health"},
    {"family": "Traffic", "dataset": "bus stops, metro stations, metro exits", "use": "public transport access"},
    {"family": "AOI", "dataset": "Baidu AOI polygons", "use": "supplement amenities, parks, housing-price proxy"},
    {"family": "Green space", "dataset": "AI interpreted green-space polygons", "use": "green/outdoor and environmental quality"},
    {"family": "Land use", "dataset": "OSM/AOI landuse parks", "use": "open space supplement"},
    {"family": "Road network", "dataset": "shanghai-roads-simplified.parquet", "use": "cycling environment and road context"},
])
source_log


# I. Inspect raw vector layers


In [ ]:
def vector_info(label, path):
    info = pyogrio.read_info(path)
    return {
        "label": label,
        "path": str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path),
        "features": info.get("features"),
        "geometry_type": info.get("geometry_type"),
        "crs": str(info.get("crs")),
        "fields": ", ".join(list(info.get("fields"))[:14]),
    }

inventory = []
for label, path in paths.items():
    if str(path).lower().endswith(".shp"):
        inventory.append(vector_info(label, path))
    else:
        inventory.append({
            "label": label,
            "path": str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path),
            "features": None,
            "geometry_type": "parquet",
            "crs": "",
            "fields": "",
        })

inventory_df = pd.DataFrame(inventory)
inventory_df


In [ ]:
inventory_df[["label", "features", "geometry_type", "crs"]].style.format({"features": "{:,.0f}"})


## I.1. Boundary and built-area validation


In [ ]:
city = gpd.read_file(paths["city_boundary"], encoding="utf-8")
districts = gpd.read_file(paths["district_boundary"], encoding="utf-8")
built = gpd.read_file(paths["built_area_2020"], encoding="utf-8")

pd.DataFrame([
    {"layer": "city", "rows": len(city), "crs": str(city.crs), "geom": city.geom_type.iloc[0], "bounds": tuple(round(x, 4) for x in city.total_bounds)},
    {"layer": "districts", "rows": len(districts), "crs": str(districts.crs), "geom": districts.geom_type.iloc[0], "bounds": tuple(round(x, 4) for x in districts.total_bounds)},
    {"layer": "built_2020", "rows": len(built), "crs": str(built.crs), "geom": built.geom_type.iloc[0], "bounds": tuple(round(x, 4) for x in built.total_bounds)},
])


In [ ]:
district_name_col = next(c for c in ["县级", "县", "NAME", "name"] if c in districts.columns)
districts[[district_name_col, "面积", "人口"]].head(10)


# II. Inspect POI files


In [ ]:
poi_dir = Path(os.environ.get("SHANGHAI_POI_SHP_DIR", ""))
print("POI directory:", poi_dir)
print("Exists:", poi_dir.exists())


In [ ]:
poi_2024_groups, poi_2024_meta = load_poi_2024_points(RAW, PROJECT_CRS)
poi_2024_meta


In [ ]:
pd.Series(poi_2024_meta.get("group_row_counts", {})).sort_values(ascending=False).to_frame("rows")


In [ ]:
sport = poi_2024_groups["sport"][0] if poi_2024_groups["sport"] else None
healthy_food = poi_2024_groups["healthy_food"][0] if poi_2024_groups["healthy_food"] else None
education = poi_2024_groups["education"][0] if poi_2024_groups["education"] else None

if sport is not None:
    display(sport[[c for c in ["name", "bigType", "midType", "smallType", "typecode", "adname"] if c in sport.columns]].head(8))
if healthy_food is not None:
    display(healthy_food[[c for c in ["name", "bigType", "midType", "smallType", "typecode", "adname"] if c in healthy_food.columns]].head(8))
if education is not None:
    display(education[[c for c in ["name", "bigType", "midType", "smallType", "typecode", "adname"] if c in education.columns]].head(8))


In [ ]:
poi_rows = []
if poi_dir.exists():
    for shp in sorted(poi_dir.glob("*.shp")):
        info = pyogrio.read_info(shp)
        parts = shp.stem.split("_")
        major = parts[1] if len(parts) >= 2 else shp.stem
        poi_rows.append({
            "major_category": major,
            "file": shp.name,
            "features": info.get("features"),
            "geometry_type": info.get("geometry_type"),
            "crs": str(info.get("crs")),
            "fields": ", ".join(list(info.get("fields"))[:10]),
        })

poi_inventory = pd.DataFrame(poi_rows).sort_values("features", ascending=False)
poi_inventory


In [ ]:
selected_poi = [
    "交通设施服务", "体育休闲服务", "公共设施", "医疗保健服务", "政府机构及社会团体",
    "生活服务", "科教文化服务", "购物服务", "风景名胜", "餐饮服务",
]
poi_inventory.assign(
    selected_for_model=poi_inventory["major_category"].isin(selected_poi)
).groupby("selected_for_model")["features"].sum().rename("features").to_frame()


In [ ]:
def read_poi_sample(major_category, n=5):
    match = poi_inventory.loc[poi_inventory["major_category"].eq(major_category), "file"].iloc[0]
    gdf = gpd.read_file(poi_dir / match, rows=n, encoding="utf-8")
    cols = [c for c in ["name", "type", "address", "经度", "纬度", "adname", "行业大", "行业中", "行业小"] if c in gdf.columns]
    return gdf[cols]

read_poi_sample("体育休闲服务", n=8)


In [ ]:
read_poi_sample("医疗保健服务", n=8)


# III. Inspect traffic, AOI, green, and land-use layers


In [ ]:
traffic_layers = {
    "bus_stops": paths["bus_stops"],
    "metro_stations": paths["metro_stations"],
    "metro_exits": paths["metro_exits"],
}

traffic_inventory = pd.DataFrame([vector_info(label, path) for label, path in traffic_layers.items()])
traffic_inventory[["label", "features", "geometry_type", "crs", "fields"]]


In [ ]:
aoi = gpd.read_file(paths["aoi"], rows=8, encoding="utf-8")
landuse = gpd.read_file(paths["landuse_webmap"], rows=8, encoding="utf-8")

display(aoi[[c for c in ["name", "type", "type1", "type2", "type3", "价格"] if c in aoi.columns]])
display(landuse[[c for c in ["name", "fclass", "code"] if c in landuse.columns]])


In [ ]:
green_layers = []
for shp in sorted((RAW / "ai_interpreted").rglob("*绿地.shp")):
    info = pyogrio.read_info(shp)
    green_layers.append({
        "file": str(shp.relative_to(ROOT)),
        "features": info.get("features"),
        "geometry_type": info.get("geometry_type"),
        "crs": str(info.get("crs")),
    })

green_inventory = pd.DataFrame(green_layers)
green_inventory.assign(features=lambda d: d["features"].map(lambda x: f"{x:,.0f}"))


# IV. Inspect road parquet, following the network-extract notebook


In [ ]:
roads_path = paths["roads"]
pq_file = pq.ParquetFile(roads_path)
roads_schema = pd.DataFrame({
    "field": pq_file.schema.names,
    "type": [str(pq_file.schema_arrow.field(i).type) for i in range(len(pq_file.schema.names))],
})
roads_schema


In [ ]:
roads = gpd.read_parquet(roads_path)
roads = roads.set_crs(PROJECT_CRS, allow_override=True) if roads.crs is None else roads.to_crs(PROJECT_CRS)
roads["length_m"] = roads.geometry.length

road_summary = pd.Series({
    "rows": len(roads),
    "total_km": roads["length_m"].sum() / 1000,
    "median_edge_m": roads["length_m"].median(),
    "bike_edges": (pd.to_numeric(roads.get("bicycle", 0), errors="coerce").fillna(0) > 0).sum(),
    "foot_edges": (pd.to_numeric(roads.get("foot", 0), errors="coerce").fillna(0) > 0).sum(),
}).round(2)
road_summary


In [ ]:
roads[["osm_id", "highway", "level", "bicycle", "foot", "length_m"]].head(10)


# V. Model design and validation rules


In [ ]:
weights = pd.DataFrame({
    "baseline_weight": pd.Series(BASELINE_WEIGHTS),
    "track_a_weight": pd.Series(TRACK_WEIGHTS),
}).fillna("")

mode_radii = pd.DataFrame(
    [{"mode": k, "radius_m": v, "radius_km": v / 1000} for k, v in MODE_RADII_M.items()]
)

display(weights)
display(mode_radii)


In [ ]:
validation_checks = pd.DataFrame([
    {"check": "All core shapefiles exist", "status": all(path.exists() for path in paths.values())},
    {"check": "POI 2024 classified CSVs loaded", "status": poi_2024_meta.get("poi_status") == "loaded"},
    {"check": "Legacy POI directory exists", "status": poi_dir.exists()},
    {"check": "2024 district layer has 16 districts", "status": len(districts) == 16},
    {"check": "Road parquet has geometry", "status": "geometry" in roads.columns},
    {"check": "POI 2024 sport rows available", "status": int(poi_2024_meta.get("group_row_counts", {}).get("sport", 0)) > 10000},
])
validation_checks


In [ ]:
out = ROOT / "data" / "processed" / "source_inventory.csv"
source_inventory = pd.concat(
    [
        inventory_df.assign(source_family="project_raw_layers"),
        poi_inventory.rename(columns={"file": "path", "major_category": "label"}).assign(source_family="gaode_poi"),
        pd.DataFrame(poi_2024_meta.get("files_used", [])).assign(label="POI 2024 classified CSVs", geometry_type="table", source_family="poi_2024"),
        green_inventory.rename(columns={"file": "path"}).assign(label="ai_green_space", source_family="ai_interpreted"),
    ],
    ignore_index=True,
    sort=False,
)
source_inventory.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Wrote {out}")
source_inventory[["source_family", "label", "features", "geometry_type"]].head(25)


## Data limitations recorded for the final transparency panel

The analysis is source-rich but still approximate. POI data indicate the existence of facilities, not quality, price, opening hours, or public accessibility. The 15-minute catchments are mode-specific distance proxies rather than GTFS or traffic-aware routes. The road parquet makes it possible to discuss cycling and network structure, but a full all-grid Dijkstra routing computation would require more runtime and a complete routable multimodal network. These limitations are carried into `summary.json` and the web app transparency panel.
